# CLEAN DATA: step by step
Run top to bottom. Change only lines with `# <<<`. Skip a step if the data does not need it.
At the end you get `../exam_clean.csv` → put it in Q1 / Q2.

### Step 1: load

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../demo_dirty.csv")        # <<< your file
# df = pd.read_excel("../exam.xlsx")         # excel
# df = pd.read_csv("../exam.txt", sep=None, engine="python")   # unknown separator
target = "Price"                              # <<< what to predict

### Step 2: look (find the problems)

In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df.head(10)

(84, 7)
ID         int64
Owner        str
Size     float64
Rooms    float64
City         str
Sold         str
Price        str
dtype: object
ID        0
Owner     0
Size      0
Rooms     3
City     18
Sold      0
Price     1
dtype: int64


,ID,Owner,Size,Rooms,City,Sold,Price
0,1,person 0,117.0,1.0,bkk,no,"$2,357,524"
1,2,person 1,191.6,4.0,Cnx,no,"$4,886,064"
2,3,person 2,54.5,2.0,Cnx,Yes,"$1,249,780"
3,4,person 3,-999.0,2.0,NaN,Yes,"$4,200,762"
4,5,person 4,83.0,4.0,cnx,yes,"$2,879,146"
5,6,person 5,102.0,NaN,cnx,no,"$3,204,306"
6,7,person 6,170.7,5.0,Cnx,Yes,"$4,354,525"
7,8,person 7,99.6,2.0,bkk,Yes,?
8,9,person 8,123.4,5.0,bkk,no,"$3,430,446"
9,10,person 9,34.7,5.0,Cnx,no,"$2,073,468"


### Step 3: drop useless columns (ID, names, free text)

In [3]:
df = df.drop(columns=["ID", "Owner"])         # <<<

### Step 4: drop duplicate rows

In [4]:
print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

duplicates: 4


### Step 5: bad values → missing

In [5]:
df = df.replace(["?", "NA", "-", "", -999], np.nan)   # <<< add what you saw in step 2

### Step 6: text that should be numbers ("$1,234", "45%")

In [6]:
for c in ["Price"]:                           # <<<
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,% ]", "", regex=True), errors="coerce")

### Step 7: fix text (" BKK" and "bkk " → "bkk")

In [7]:
for c in ["City", "Sold"]:                    # <<< text columns
    df[c] = df[c].str.strip().str.lower()
print(df["City"].value_counts(dropna=False))

City
cnx    32
bkk    31
NaN    17
Name: count, dtype: int64


### Step 8: missing values

In [8]:
df = df.dropna(subset=[target])               # no target = useless row
for c in df.columns:
    if df[c].isna().any():
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(df[c].median())
        else:
            df[c] = df[c].fillna(df[c].mode()[0])
print(df.isna().sum().sum(), "missing left")

0 missing left


### Step 9 (only if asked): remove outliers

In [9]:
c = "Size"                                    # <<<
q1, q3 = df[c].quantile([0.25, 0.75])
low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print("outliers:", ((df[c] < low) | (df[c] > high)).sum())
# df = df[(df[c] >= low) & (df[c] <= high)]  # uncomment to remove

outliers: 0


### Step 10: text → numbers

In [10]:
df["Sold"] = df["Sold"].map({"yes": 1, "no": 0})          # <<< 2 values: map
df = pd.get_dummies(df, columns=["City"], drop_first=True, dtype=int)   # <<< 3+ values: one-hot

### Step 11: check + save

In [11]:
print(df.dtypes)
print(df.shape, "| missing:", df.isna().sum().sum())
df.to_csv("../exam_clean.csv", index=False)
print("columns for cols =", [c for c in df.columns if c != target])
df.head()

Size        float64
Rooms       float64
Sold          int64
Price       float64
City_cnx      int64
dtype: object
(77, 5) | missing: 0
columns for cols = ['Size', 'Rooms', 'Sold', 'City_cnx']


,Size,Rooms,Sold,Price,City_cnx
0,117.0,1.0,0,2357524.0,0
1,191.6,4.0,0,4886064.0,1
2,54.5,2.0,1,1249780.0,1
3,121.5,2.0,1,4200762.0,1
4,83.0,4.0,1,2879146.0,1


### Step 12: use it
In Q1 / Q2 cell 1:
```python
df = pd.read_csv("../exam_clean.csv")
cols = ["Size", "Rooms", "City_cnx"]
target = "Price"
```